# População por estado e por município

A planilha do Censo 2022 compara a população municipal de 2010, já ajustada às alterações de limite até 2022, com a população do Censo 2022. Cada linha é um município. Não há linha de total do Brasil para somar de novo.

A diferença pedida é 2022 menos 2010. Uso a coluna “População 2010 (Alterações de Limites até 2022)”, não a sinopse, porque o arquivo é a comparação compatibilizada.

Execute a partir da pasta da atividade ou da raiz de `analise_dados`.


In [1]:
from pathlib import Path

import pandas as pd

def raiz_do_repositorio():
    aqui = Path.cwd().resolve()
    for pasta in [aqui, *aqui.parents]:
        if (pasta / "dados").is_dir() and (pasta / "etapa_3").is_dir():
            return pasta
    raise FileNotFoundError("Execute a partir do repositorio analise_dados.")

RAIZ = raiz_do_repositorio()
DADOS = RAIZ / "dados"
SAIDAS = RAIZ / "etapa_3" / "03_populacao" / "saidas"
SAIDAS.mkdir(parents=True, exist_ok=True)

arquivo = DADOS / "CD2022_Populacao_2010_Compatibilizada_20231222.xlsx"
bruto = pd.read_excel(arquivo, sheet_name="Municípios", header=None)
print(bruto.shape)
bruto.head(4)


(5577, 8)


,0,1,2,3,4,5,6,7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Censo Demográfico 2022: População e Domicílios...,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População Município 2010\n(Sinopse),População 2010 (Alterações de Limites até 2022)1,População Censo 2022
3,NaN,RO,11,00015,Alta Floresta D'Oeste,24392,24392,21494


In [2]:
mascara = bruto.apply(
    lambda coluna: coluna.astype(str).str.contains("NOME DO MUNIC", na=False)
)
linha_cabecalho = int(mascara.any(axis=1).idxmax())
cabecalho = bruto.iloc[linha_cabecalho]

colunas = {}
for indice, texto in cabecalho.items():
    if pd.isna(texto):
        continue
    nome = str(texto).replace("\n", " ")
    if nome.strip() == "UF":
        colunas["uf"] = indice
    elif "COD. UF" in nome:
        colunas["cod_uf"] = indice
    elif "COD. MUNIC" in nome:
        colunas["cod_munic"] = indice
    elif "NOME DO MUNIC" in nome:
        colunas["municipio"] = indice
    elif "Sinopse" in nome:
        colunas["pop_2010_sinopse"] = indice
    elif "Alterações de Limites" in nome or "Alteracoes de Limites" in nome:
        colunas["pop_2010"] = indice
    elif "2022" in nome:
        colunas["pop_2022"] = indice

print("cabecalho na linha", linha_cabecalho)
print(colunas)


cabecalho na linha

 2
{'uf': 1, 'cod_uf': 2, 'cod_munic': 3, 'municipio': 4, 'pop_2010_sinopse': 5, 'pop_2010': 6, 'pop_2022': 7}


In [3]:
siglas = [
    "RO", "AC", "AM", "RR", "PA", "AP", "TO",
    "MA", "PI", "CE", "RN", "PB", "PE", "AL", "SE", "BA",
    "MG", "ES", "RJ", "SP",
    "PR", "SC", "RS",
    "MS", "MT", "GO", "DF",
]

tabela = bruto.iloc[linha_cabecalho + 1 :, list(colunas.values())].copy()
tabela.columns = list(colunas.keys())
tabela = tabela[tabela["uf"].isin(siglas)].copy()

for coluna in ["pop_2010_sinopse", "pop_2010", "pop_2022"]:
    tabela[coluna] = pd.to_numeric(tabela[coluna], errors="coerce")

tabela["cod_uf"] = tabela["cod_uf"].astype(int).astype(str).str.zfill(2)
tabela["cod_munic"] = (
    tabela["cod_munic"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
)
tabela["codigo_ibge"] = tabela["cod_uf"] + tabela["cod_munic"]
tabela["diferenca_2022_2010"] = tabela["pop_2022"] - tabela["pop_2010"]

print("municipios:", len(tabela))
print("ufs:", tabela["uf"].nunique())
print("codigos duplicados:", int(tabela["codigo_ibge"].duplicated().sum()))
print("populacao ausente:", tabela[["pop_2010", "pop_2022"]].isna().sum().to_dict())
print(
    "municipios com 2010 da sinopse diferente do 2010 compatibilizado:",
    int((tabela["pop_2010_sinopse"] != tabela["pop_2010"]).sum()),
)


municipios: 5570
ufs: 27
codigos duplicados: 0
populacao ausente: {'pop_2010': 0, 'pop_2022': 0}
municipios com 2010 da sinopse diferente do 2010 compatibilizado: 1623


In [4]:
por_estado = (
    tabela.groupby("uf", as_index=False)[["pop_2010", "pop_2022"]]
    .sum()
)
por_estado["diferenca_2022_2010"] = por_estado["pop_2022"] - por_estado["pop_2010"]
por_estado = por_estado.sort_values(
    ["diferenca_2022_2010", "uf"], ascending=[False, True]
).reset_index(drop=True)

print("soma dos estados confere com a soma dos municipios?")
print(por_estado[["pop_2010", "pop_2022"]].sum().tolist() == [
    tabela["pop_2010"].sum(),
    tabela["pop_2022"].sum(),
])
por_estado


soma dos estados confere com a soma dos municipios?


True


,uf,pop_2010,pop_2022,diferenca_2022_2010
0,SP,41262199,44411238,3149039
1,SC,6248436,7610361,1361925
2,GO,6001789,7056495,1054706
3,PR,10444526,11444380,999854
4,MG,19597330,20539989,942659
5,MT,3035122,3658649,623527
6,PA,7581051,8120131,539080
7,AM,3483985,3941613,457628
8,CE,8451644,8794957,343313
9,ES,3514952,3833712,318760


In [5]:
por_municipio = tabela[
    [
        "uf",
        "cod_uf",
        "cod_munic",
        "codigo_ibge",
        "municipio",
        "pop_2010",
        "pop_2022",
        "diferenca_2022_2010",
    ]
].sort_values(
    ["diferenca_2022_2010", "municipio", "uf"],
    ascending=[False, True, True],
).reset_index(drop=True)

nomes_repetidos = por_municipio.groupby("municipio")["uf"].nunique()
nomes_repetidos = nomes_repetidos[nomes_repetidos > 1]
print("nomes de municipio que aparecem em mais de um estado:", len(nomes_repetidos))
print(por_municipio.head(10))


nomes de municipio que aparecem em mais de um estado: 232
   uf cod_uf cod_munic codigo_ibge      municipio  pop_2010  pop_2022  \
0  AM     13     02603     1302603         Manaus   1802014   2063689   
1  DF     53     00108     5300108       Brasília   2572159   2817381   
2  SP     35     50308     3550308      São Paulo  11253503  11451999   
3  SP     35     52205     3552205       Sorocaba    586816    723682   
4  GO     52     08707     5208707        Goiânia   1301912   1437366   
5  RR     14     00100     1400100      Boa Vista    284313    413486   
6  SC     42     05407     4205407  Florianópolis    421240    537211   
7  PA     15     05536     1505536    Parauapebas    153908    267836   
8  MS     50     02704     5002704   Campo Grande    786774    898100   
9  PB     25     07507     2507507    João Pessoa    723515    833932   

   diferenca_2022_2010  
0               261675  
1               245222  
2               198496  
3               136866  
4            

In [6]:
arquivo_estado = SAIDAS / "populacao_por_estado.csv"
arquivo_municipio = SAIDAS / "populacao_por_municipio.csv"
por_estado.to_csv(arquivo_estado, sep=";", index=False)
por_municipio.to_csv(arquivo_municipio, sep=";", index=False)

estado_lido = pd.read_csv(arquivo_estado, sep=";")
municipio_lido = pd.read_csv(arquivo_municipio, sep=";")
print(estado_lido.columns.tolist(), len(estado_lido))
print(municipio_lido.columns.tolist(), len(municipio_lido))
print("estado ordenado:", estado_lido["diferenca_2022_2010"].is_monotonic_decreasing)
print("municipio ordenado:", municipio_lido["diferenca_2022_2010"].is_monotonic_decreasing)
estado_lido.head()

['uf', 'pop_2010', 'pop_2022', 'diferenca_2022_2010'] 27
['uf', 'cod_uf', 'cod_munic', 'codigo_ibge', 'municipio', 'pop_2010', 'pop_2022', 'diferenca_2022_2010'] 5570
estado ordenado: True
municipio ordenado: True


,uf,pop_2010,pop_2022,diferenca_2022_2010
0,SP,41262199,44411238,3149039
1,SC,6248436,7610361,1361925
2,GO,6001789,7056495,1054706
3,PR,10444526,11444380,999854
4,MG,19597330,20539989,942659
